In [1]:
import vectorbt as vbt
import yfinance as yf
import pandas_ta as ta
import pandas as pd
import numpy as np
import itertools
from sklearn.model_selection import TimeSeriesSplit

# Pull data using the clean Ticker method
spy = yf.Ticker('SPY')
# We isolate just the 'Close' column as a Pandas Series
price_data = spy.history(start='2020-01-01', end='2024-01-01')[['Open', 'High', 'Low', 'Close']]

# Generate a 14-day RSI

fast_windows = np.arange(5, 60, step=5)
# Slow EMAs: 100, 120, 140, 160, 180, 200
slow_windows = np.arange(80, 220, step=10)

combinations = list(itertools.product(fast_windows, slow_windows))
fast_list = [c[0] for c in combinations]
slow_list = [c[1] for c in combinations]

tscv = TimeSeriesSplit(n_splits = 5)

oos_returns = []

adx_df = ta.adx(price_data['High'], price_data['Low'], price_data['Close'], length=14)
adx_series = adx_df['ADX_14']

for fold, (train_index, test_index) in enumerate(tscv.split(price_data)):

    train_data = price_data.iloc[train_index]
    test_data = price_data.iloc[test_index]

    train_start, train_end = train_data.index.min().date(), train_data.index.max().date()
    test_start, test_end = test_data.index.min().date(), test_data.index.max().date()


    Ex_Fast_train = vbt.MA.run(train_data['Close'], window = fast_list, ewm = True
                               )
    Ex_Slow_train = vbt.MA.run(train_data['Close'], window = slow_list, ewm = True)

    # Slice the ADX for the training window
    train_adx = adx_series.iloc[train_index]

    # Create a boolean mask: True if ADX > 25, False otherwise
    is_trending_train = train_adx > 25

    # Generate original crossovers
    base_entries_train = Ex_Fast_train.ma_crossed_above(Ex_Slow_train)
    exits_train = Ex_Fast_train.ma_crossed_below(Ex_Slow_train)

    # NEW LOGIC: Only allow the entry if a crossover happened AND the market is trending
    # We use .mul(axis=0) to correctly broadcast the 1D ADX series across the 154-column matrix
    entries_train = base_entries_train.mul(is_trending_train, axis=0).astype(bool)

    multi_index = pd.MultiIndex.from_tuples(combinations, names=['fast', 'slow'])
    entries_train.columns = multi_index
    exits_train.columns = multi_index



    safe_entries_train = entries_train.shift(1).fillna(False).astype(bool)
    safe_exits_train = exits_train.shift(1).fillna(False).astype(bool)

    portfolio_train = vbt.Portfolio.from_signals(
        close = train_data['Open'],     # <--- EXECUTING AT THE OPEN
        entries = safe_entries_train,
        exits = safe_exits_train,
        init_cash=10000,
        fees=0.001,
        slippage=0.0001,
        freq = '1D',
        sl_stop=0.15,     # The 5% stop-loss threshold
        sl_trail=True
    )

    best_fast, best_slow = portfolio_train.total_return().idxmax()

    Ex_Fast_test = vbt.MA.run(test_data['Close'], window = best_fast, ewm = True)
    Ex_Slow_test = vbt.MA.run(test_data['Close'], window = best_slow, ewm = True)

    test_adx = adx_series.iloc[test_index]
    is_trending_test = test_adx > 25

    # Apply to the out-of-sample test
    base_entries_test = Ex_Fast_test.ma_crossed_above(Ex_Slow_test)
    exit_test = Ex_Fast_test.ma_crossed_below(Ex_Slow_test)

    entries_test = base_entries_test.mul(is_trending_test, axis=0).astype(bool)

    safe_entries_test = entries_test.shift(1).fillna(False).astype(bool)
    safe_exits_test = exit_test.shift(1).fillna(False).astype(bool)

    pf_test = vbt.Portfolio.from_signals(
        close=test_data['Open'], entries=safe_entries_test, exits=safe_exits_test,
        fees=0.001, slippage=0.0001, freq='1D', sl_stop=0.15, sl_trail=True
    )
    # Extract the raw return
    raw_return = pf_test.total_return()

    # If vectorbt wrapped it in a Series, extract the raw float. Otherwise, leave it alone.
    if isinstance(raw_return, pd.Series):
        raw_return = raw_return.iloc[0]

    oos_return = raw_return * 100
    oos_returns.append(oos_return)

    print(f"Fold {fold + 1}:")
    print(pf_test.stats())
    print(f"  Train ({train_start} to {train_end}) | Best Params: {best_fast}/{best_slow}")
    print(f"  Test  ({test_start} to {test_end}) | OOS Return: {oos_return:>6.2f}%\n")

Fold 1:
Start                         2020-09-04 00:00:00-04:00
End                           2021-05-05 00:00:00-04:00
Period                                167 days 00:00:00
Start Value                                       100.0
End Value                                         100.0
Total Return [%]                                    0.0
Benchmark Return [%]                          21.981723
Max Gross Exposure [%]                              0.0
Total Fees Paid                                     0.0
Max Drawdown [%]                                    NaN
Max Drawdown Duration                               NaT
Total Trades                                          0
Total Closed Trades                                   0
Total Open Trades                                     0
Open Trade PnL                                      0.0
Win Rate [%]                                        NaN
Best Trade [%]                                      NaN
Worst Trade [%]                         

In [2]:

# Calculate the true expectancy of your algorithm
print(f"--- Final OOS Average Return across all 5 folds: {np.mean(oos_returns):.2f}% ---")

--- Final OOS Average Return across all 5 folds: 3.28% ---
